# Artitech: Adaptive Artwork Recommendation System

**Goal:** This notebook demonstrates an adaptive Retrieval-Augmented Generation (RAG) system combined with an artwork recommender. The system aims to provide empathetic responses and relevant artwork suggestions based on user input and detected emotions.

**Core Components:**
*   **Artwork Recommender:** Uses CLIP embeddings and color analysis to find artworks related to emotions. (`src/artwork_recommender.py`, `src/recommender_setup.py`)
*   **Adaptive RAG Graph:** Uses LangGraph to dynamically route user queries based on context, relevance, and emotion. (`src/graph_builder.py`, `src/graph_nodes.py`, `src/emotion_nodes.py`)
*   **Data Processing:** Loads and prepares text data (PDFs) and artwork data (CSV, images). (`src/data_processing.py`)
*   **LangChain Components:** Leverages various LangChain elements for prompting, retrieval, grading, and chaining. (`src/chain.py`, `src/prompts.py`, `src/grading.py`, etc.)

## 1. Setup and Initialization

This section handles importing necessary libraries, setting up API keys, and configuring application settings.

In [ ]:
# 필요한 패키지 설치
!pip install -q langchain langchain-community langchain-openai
!pip install -q chromadb
!pip install -q python-dotenv
!pip install -q langgraph
!pip install -q tavily-python
!pip install -q pypdf
!pip install -q langchain-core
!pip install -q langchain-experimental
!pip install -q pandas numpy
!pip install -q langchain-huggingface
!pip install -q langchain-elasticsearch
!pip install -q graphviz
!pip install -q langchain-teddynote
!pip install --upgrade --force-reinstall urllib3
!pip install -q faiss-cpu


ERROR: Could not find a version that satisfies the requirement langchain-teddynote (from versions: none)
ERROR: No matching distribution found for langchain-teddynote


  Using cached urllib3-2.4.0-py3-none-any.whl.metadata (6.5 kB)
Using cached urllib3-2.4.0-py3-none-any.whl (128 kB)
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.4.0
    Uninstalling urllib3-2.4.0:
      Successfully uninstalled urllib3-2.4.0
  Using cached accelerate-1.6.0-py3-none-any.whl.metadata (19 kB)


ERROR: Could not find a version that satisfies the requirement intel-extension-for-pytorch (from versions: none)
ERROR: No matching distribution found for intel-extension-for-pytorch


### 1.1 API Key Configuration

We need to set up API keys for OpenAI and potentially other services like Tavily Search. The `fix_api_key.py` script helps manage this process, attempting to load keys from a `.env` file or using predefined values if necessary.

In [ ]:
import os
from dotenv import load_dotenv

os.chdir("artipexllm/artipexllm-JH_dev") # 현재 작업환경
print(f"Current working directory: {os.getcwd()}")
print("Current working directory:", os.getcwd())
print("Files in directory:", os.listdir())

load_dotenv()

Current working directory: C:\university\sjtu\2024-2025 1\LLM\team\artipexllm\artipexllm-JH_dev
Current working directory: C:\university\sjtu\2024-2025 1\LLM\team\artipexllm\artipexllm-JH_dev
Files in directory: ['.env', '.gitattributes', '.gitignore', '.ipynb_checkpoints', 'adaptive_chat_interface.py', 'artitech_final.ipynb', 'artitech_final_YC.ipynb', 'Artitech_LangGraph.png', 'chain_YC.py', 'check.py', 'data', 'downloads', 'emotion_art_cache', 'graph.py', 'LICENSE', 'README.md', 'README_kor.md', 'requirements.txt', 'scripts', 'search_paper.py', 'src', '자료']


True

In [2]:
# fix_api_key.py에서 함수 가져오기
from scripts.fix_api_key import get_api_key_from_env_file
from scripts.fix_api_key import fix_tavily_api_key

# OpenAI API 키 설정
openai_api_key = get_api_key_from_env_file(key_name="OPENAI_API_KEY")

# Tavily API 키 설정 (필요한 경우)
tavily_api_key = fix_tavily_api_key()


Loaded environment variables from: C:\university\sjtu\2024-2025 1\LLM\team\artipexllm\artipexllm-JH_dev\.env
✓ Found valid OPENAI_API_KEY in environment/.env: sk-pr...GK4A
Loaded environment variables from: C:\university\sjtu\2024-2025 1\LLM\team\artipexllm\artipexllm-JH_dev\.env
✓ Found valid TAVILY_API_KEY in environment/.env: tvly-...
Tavily API Key loaded and set: tvly-...


In [3]:
LANGSMITH_PROJECT = "ArtiTech_test"

In [4]:
from langchain_teddynote import logging

# Start tracking the langsmith
logging.langsmith("ArtiTech_test")

LangSmith 추적을 시작합니다.
[프로젝트명]
ArtiTech_test


## 2. Data Loading and Preparation

Here, we load the necessary data for both the RAG system (text documents) and the artwork recommender (artwork metadata and images).

### 2.1 Loading and Splitting Text Documents

We use functions from `src/data_processing.py` to load PDF documents from a specified directory. These documents are then split into smaller chunks using an adaptive strategy based on document length. This is crucial for efficient retrieval in the RAG system.

### 2.2 Initializing Embeddings Model

We need an embedding model (e.g., from OpenAI) to convert text chunks into vector representations for similarity search.

In [ ]:
# Cell: Load Data and Create Retriever
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from src.data_processing import load_and_split_pdfs
from src.retriever import create_ensemble_retriever
from huggingface_hub import login

login("hf_") # 본인 huggingface API
# 데이터 디렉토리 설정
DATA_DIR = "data/RAG_pdf"

# 1. Load and split documents
split_documents = load_and_split_pdfs(DATA_DIR)

# 2. Initialize embeddings
embeddings = OpenAIEmbeddings()
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Total 26 pdf files found.
'1.psychological-effects-of-colour.pdf'에서 2 페이지를 로드했습니다.
'10.Colour_Psychology_in_Art.pdf'에서 8 페이지를 로드했습니다.
'11.Study_on_Color_Art_Therapy_Techniques.pdf'에서 21 페이지를 로드했습니다.
'12.Visual_Symbolization.pdf'에서 9 페이지를 로드했습니다.
'13.Art therapy and neuroscience.pdf'에서 6 페이지를 로드했습니다.
'14.Formal_Elements_of_Art_Products.pdf'에서 13 페이지를 로드했습니다.
'2.Final-Over-60-Art-Therapy-Exercises.pdf'에서 62 페이지를 로드했습니다.
'3.Color_Psychology_Effects_of_Perceiving_Color_on_Psychological_Functioning_in_Humans.pdf'에서 29 페이지를 로드했습니다.
'4.Art_and_Design_Foundation_LM_Section-6_LV.pdf'에서 26 페이지를 로드했습니다.
'5.Understanding_Formal_Analysis.pdf'에서 3 페이지를 로드했습니다.
'6.Elements_of_art.pdf'에서 3 페이지를 로드했습니다.
'7.verywellmind.com.pdf'에서 4 페이지를 로드했습니다.
'8.Brushing Away Stress_ 21 Art Therapy Activities for Self-Expression and Healing - RMCAD.pdf'에서 11 페이지를 로드했습니다.
'An Investigation into Art Therapy Aided Health and Well-Being  Research A  75-Year Bibliometric Analysis.pdf'에서 27 페이지를 로드했습니다.
파일 'Art therapy is 

## 3. Retrieval-Augmented Generation (RAG) Components

Now, we set up the individual components required for the adaptive RAG workflow. These components will be assembled into a graph later.

### 3.1 Creating the Retriever

We create an `EnsembleRetriever` (using `src/retriever.py`) which combines:
1.  **FAISS Retriever:** For dense vector similarity search based on semantic meaning.
2.  **BM25 Retriever:** For sparse, keyword-based search.

This hybrid approach often yields better results than using either method alone.

In [6]:
# 3. Create the ensemble retriever
# Ensure split_documents is not empty before proceeding
if split_documents:
    ensemble_retriever = create_ensemble_retriever(split_documents, embeddings, k=5)
else:
    print("Error: No documents were loaded or split. Cannot create retriever.")
    # Handle the error appropriately, maybe raise an exception or exit
    ensemble_retriever = None

Ensemble (Hybrid) retriever created.


In [8]:
# Cell: Test Retriever (Optional)
if ensemble_retriever:
    test_query = "I really had a bad day today. What color can heal my mood?"
    # Use invoke instead of get_relevant_documents
    test_retrieved_docs = ensemble_retriever.invoke(test_query)
    print(f"Retrieved {len(test_retrieved_docs)} documents for test query:")
    for i, doc in enumerate(test_retrieved_docs):
        print(f"--- Doc {i+1} (Source: {doc.metadata.get('source', 'N/A')}, Page: {doc.metadata.get('page', 'N/A')}) ---")
        print(doc.page_content[:200] + "...") # Print start of content
    else:
        print("Retriever not created, skipping test.")

Retrieved 10 documents for test query:
--- Doc 1 (Source: data/RAG_pdf\ATLAS OF COLORS.pdf, Page: 21) ---
med doors, an aggressive behavior. 
 • Suitable color hues
Based on a TID approach, the fight mode needs a safe release option to 
soothe these violent reactions. For instance, the possibility of exer...
--- Doc 2 (Source: data/RAG_pdf\7.verywellmind.com.pdf, Page: 3) ---
1. Carsley D, Heath N. Eﬀectiveness of mindfulness-based colouring for test anxiety in
adolescents. Sch Psychol Int. 2018;39(3):251-272. doi:10.1177/0143034318773523
2. Bell, Chloe E.; Robbins, Steven...
--- Doc 3 (Source: data/RAG_pdf\10.Colour_Psychology_in_Art.pdf, Page: 1) ---
A. R. Hussain 
 
 
DOI: 10.4236/adr.2021.94025 302 Art and Design Review 
 
steeped in colour, shaping how we interpret matter (particles and fields). Colour 
continually surrounds us and mediates our...
--- Doc 4 (Source: data/RAG_pdf\ATLAS OF COLORS.pdf, Page: 2) ---
ficant and makes it a powerful tool that can be used to achieve desir

### 3.2 Defining Prompts

Prompts guide the behavior of the LLM. We define several key prompts in `src/prompts.py`:
*   `prompt_template`: The main conversational prompt defining the AI's persona and response structure.
*   `rewrite_prompt`: Used to rephrase user questions for better retrieval.
*   `citation_prompt`: Used for enforcing citation generation (though may not be directly used in the final graph's generation steps if the main prompt handles it).

In [1]:
from src.chain import prepare_rag_input, create_og_chain
from src.prompts import prompt_template # Import your prompt
from langchain_openai import ChatOpenAI # To define the llm

llm = ChatOpenAI(model_name="gpt-4o", temperature=0)



OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
import time
import psutil
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# -----------------------------
# A. Without IPEX
# -----------------------------
model_no_ipex = AutoModelForCausalLM.from_pretrained(model_name)
model_no_ipex.eval()

pipe_no_ipex = pipeline("text-generation", model=model_no_ipex, tokenizer=tokenizer, max_new_tokens=256, device=-1)
llm_no_ipex = HuggingFacePipeline(pipeline=pipe_no_ipex)

# -----------------------------
# B. With IPEX
# -----------------------------
import intel_extension_for_pytorch as ipex
model_ipex = AutoModelForCausalLM.from_pretrained(model_name)
model_ipex = ipex.optimize(model_ipex, dtype=torch.float32)
model_ipex.eval()

pipe_ipex = pipeline("text-generation", model=model_ipex, tokenizer=tokenizer, max_new_tokens=256, device=-1)
llm_local = HuggingFacePipeline(pipeline=pipe_ipex)


In [ ]:
def benchmark_model(pipe, prompt, label=""):
    start_time = time.time()
    cpu_before = psutil.cpu_percent(interval=1)
    result = pipe(prompt, max_new_tokens=256)[0]["generated_text"]
    cpu_after = psutil.cpu_percent(interval=1)
    duration = time.time() - start_time
    print(f"\n[{label}] 응답 예시: {result[:150]}...\n")
    print(f"[{label}] 추론 시간: {duration:.2f}초, CPU 평균 사용률: {(cpu_before + cpu_after)/2:.2f}%")
    return duration, (cpu_before + cpu_after)/2

test_prompt = "Please guide me through a simple art activity to help me relax."

# 비교 실행
benchmark_model(pipe_no_ipex, test_prompt, label="No IPEX")
benchmark_model(pipe_ipex, test_prompt, label="With IPEX")


In [ ]:
if 'llm_local' in locals() and 'prompt_template' in locals():
    og_chain_modular = create_og_chain(llm_local, prompt_template)
    print("Modular OG Chain created successfully.")

In [ ]:
# Define the Retriever (ensure 'ensemble_retriever' exists from previous cells)
if 'ensemble_retriever' in locals():
    # Prepare input using the helper function
    test_question = "Please guide me through a simple art activity to help my relax. I am interested in it."
    input_data = prepare_rag_input(test_question, ensemble_retriever) # Uses the helper from src/chain.py

    # Invoke the chain
    response = og_chain_modular.invoke(input_data)
    print("\n----- Modular OG Chain Response -----")
    print(response)
else:
    print("Error: 'ensemble_retriever' not defined. Cannot test the chain.")

### 3.3 Creating Grader Chains

Graders are LLM chains used to evaluate intermediate results within the RAG flow (`src/grading.py`):
*   **Retrieval Grader:** Assesses if a retrieved document is relevant to the question.
*   **Hallucination Grader:** Checks if the generated answer is grounded in the provided documents.
*   **Answer Grader:** Evaluates if the generated answer actually addresses the user's question.

These graders output structured data (Pydantic models) for reliable decision-making in the graph.

In [ ]:
from langchain_openai import ChatOpenAI # To define the llm

# --- Import Creator Functions ---
from src.routing import create_question_router
from src.grading import create_retrieval_grader, create_hallucination_grader, create_answer_grader
from src.rewriting import create_question_rewriter 
# ------------------------------

# --- Initialize LLM --- 
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)
# ----------------------

# --- Initialize Router and Graders --- 
question_router = create_question_router(llm)
retrieval_grader = create_retrieval_grader(llm)
hallucination_grader = create_hallucination_grader(llm)
answer_grader = create_answer_grader(llm)
question_rewriter = create_question_rewriter(llm)

print("LLM, Router, and Graders Initialized.")
# -------------------------------------

## 4. Artwork Recommender System

This section focuses on setting up the `ArtworkRecommender`, which finds relevant artworks based on emotional prompts.

### 4.1 Initializing the Recommender

The `initialize_recommender` function (from `src/recommender_setup.py`) encapsulates the setup process for the `ArtworkRecommender` class (defined in `src/artwork_recommender.py`). This involves:
*   Loading the specified CLIP model (`openai/clip-vit-large-patch14` by default).
*   Loading artwork metadata from the CSV (`config.CSV_PATH`).
*   Checking for corresponding images (`config.IMAGE_FOLDER`).
*   Generating or loading cached image embeddings (`config.CACHE_DIR`).
*   Optionally building a FAISS index for faster search if FAISS is available (`config.FAISS_AVAILABLE`).

In [ ]:
# Cell: Initialize Artwork Recommender and Load Data
import os
import sys

# Use the explicitly set working directory or dynamically find project root
project_root = '/Users/ijunhyeong/Desktop/Artitech_Final' # Please rename this with your project_root.
src_dir = os.path.join(project_root, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
print(f"Ensured '{src_dir}' is in sys.path")

# --- Import setup function and initialize ---
try:
    # --- Add reload logic if kernel restart isn't sufficient ---
    import importlib
    import src.config
    import src.recommender_setup
    importlib.reload(src.config)
    importlib.reload(src.recommender_setup)
    # --- End reload logic ---

    from src.recommender_setup import initialize_recommender
    from src.display_utils import display_recommendations # Keep display utils import separate if used later
    from src.config import IN_NOTEBOOK # Import config needed for display

    # Set rebuild_cache=True if you need to regenerate embeddings
    recommender = initialize_recommender(rebuild_cache=False)

    if recommender:
        print("Artwork Recommender initialized successfully.")
    else:
        print("Artwork Recommender initialization failed.")

except ImportError as e:
    print(f"ImportError: {e}")
    print("Please ensure 'src' directory is correct and dependencies are installed.")
    recommender = None # Ensure recommender is None if import fails
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    recommender = None # Ensure recommender is None on other errors

In [ ]:
# Example Usage (Optional)
if recommender:
     print("\\n--- Example Recommendation ---")
     test_emotion = "Love"
     recommendations_df, similarities = recommender.get_recommendations(test_emotion, top_n=3)

     if recommendations_df is not None:
         print(f"\\nDisplaying top {len(recommendations_df)} recommendations for '{test_emotion}':")
         # Decide whether to display images based on environment
         display_recommendations(recommendations_df, similarities, display_images=IN_NOTEBOOK)
     else:
         print(f"Could not get recommendations for '{test_emotion}'.")

## 5. Building the Adaptive RAG Graph

We use LangGraph to define the application's control flow as a state machine.

### 5.1 Graph State

The `GraphState` TypedDict (defined in `src/state.py`) holds all the data passed between nodes in the graph, including the question, documents, generation, conversation history, emotion details, and art recommendation status.

### 5.2 Defining Nodes and Edges

The core logic resides in the graph nodes defined in `src/graph_nodes.py` and `src/emotion_nodes.py`. The `create_adaptive_rag_graph` function in `src/graph_builder.py` assembles these nodes and defines the conditional edges that control the flow based on the current state (e.g., relevance grades, detected emotion).

Key steps in the graph include:
1.  **Routing:** Initial routing based on user input type (memory, web, vectorstore, art request).
2.  **Retrieval & Grading:** Fetching and filtering relevant documents.
3.  **Generation & Hallucination Check:** Generating an answer and ensuring it's grounded.
4.  **Emotion Detection:** Analyzing the user's query/context for emotion.
5.  **Art Suggestion/Recommendation:** Conditionally suggesting or displaying artwork based on emotion and user preference.

In [ ]:
# Cell: Define and Compile LangGraph Workflow

# Required imports for graph components (keep necessary ones here if used outside graph creation)
# from langgraph.graph import END, StateGraph, START # Moved to graph_builder
from langgraph.checkpoint.memory import MemorySaver # Needed here to create checkpointer
# from src.state import GraphState # Moved to graph_builder

# --- Import the graph builder function ---
# from src.graph_builder import create_adaptive_rag_graph # Moved below reload

# --- Create checkpointer ---
checkpointer = MemorySaver()

# --- Build the graph ---
try:
    # Ensure sys.path includes src and scripts if needed for imports within graph_builder
    # (The sys.path manipulation should ideally happen in an earlier cell)
    import importlib
    import src.graph_builder
    # Also reload dependencies if they changed
    try:
        import src.state
        importlib.reload(src.state)
    except ImportError: pass # Ignore if state module doesn't exist or fails
    try:
        import src.graph_nodes
        importlib.reload(src.graph_nodes)
    except ImportError: pass 
    # Ignore if nodes module doesn't exist or fails
    # Reloading scripts might be tricky, ensure they are stable or moved to src
    # try:
    #     import scripts.emotion.multi_turn_router
    #     importlib.reload(scripts.emotion.multi_turn_router)
    #     # ... reload other script dependencies ...
    # except ImportError: pass

    importlib.reload(src.graph_builder) # Reload the builder itself
    from src.graph_builder import create_adaptive_rag_graph # Re-import after reload

    adaptive_rag = create_adaptive_rag_graph(checkpointer=checkpointer)
    print("Graph compilation triggered from notebook.")
except ImportError as e:
    print(f"ImportError during graph creation: {e}")
    print("Check imports within src/graph_builder.py and ensure necessary modules (like src.state, src.graph_nodes, and potentially scripts) are accessible.")
    adaptive_rag = None
except Exception as e:
    print(f"An unexpected error occurred during graph creation: {e}")
    adaptive_rag = None

In [ ]:
import langgraph
from langchain_teddynote.graphs import visualize_graph

# Visualize the graph
langgraph.graph.graph.TIMEOUT = 60  # Increase to 60 seconds
visualize_graph(adaptive_rag)

## 6. Running the Application

Now we can interact with the compiled LangGraph application. We invoke it with the user's query and a configuration dictionary specifying the thread ID for maintaining conversation state.

### 6.1 Invoking the Graph

We send the input question to the graph's stream or invoke method. The `config` dictionary is crucial for tracking conversational memory using the checkpointer.

In [ ]:
# Import the utility functions for running the graph
#from graph_utils import print_graph_structure
from langchain_core.runnables import RunnableConfig
# Configure the inputs
config = RunnableConfig(recursion_limit=20, configurable={"thread_id": "test_run"})

# Test Case 1: Direct art request
inputs1 = {
    "question": "I'm feeling really sad today after receiving some bad news",
}

# Run the graph with the direct art request
print("===== Test Case 1: Direct Art Request =====")
result1 = adaptive_rag.invoke(inputs1, config)

# --- Add this part ---
# Extract the relevant messages from the result state
rag_answer = result1.get("generation", "")
suggestion_offer = result1.get("suggestion_message", "")

# Combine and print the desired output for the user
full_response = f"{rag_answer}\n\n{suggestion_offer}".strip()
print("\n--- AI Response (Turn 1) ---")
print(full_response)
print("---------------------------\n")


### 6.2 Displaying Artwork Recommendations

If the graph generated artwork recommendations (`art_recommendations` in the state), we can use the `display_recommendations` utility from `src/display_utils.py` to show them, including images (if in a notebook environment).

In [ ]:
user_response_turn_2 = "Yes, please show me one art that can make me feel better."

inputs2 = {
        "question": user_response_turn_2,
    }

print("\n===== Test Case 1: Second Turn - User Accepts Art Offer =====")
result2 = adaptive_rag.invoke(inputs2, config)
# --- Add this part ---
# Extract the relevant messages from the result state
final_response = result2.get("generation", "")
#suggestion_offer = result2.get("suggestion_message", "")

# Combine and print the desired output for the user
print("\n--- AI Response (Turn 2) ---")
print(final_response)
print("---------------------------\n")

In [ ]:
user_response_turn_3 = "If I want to draw my own art referencing this artwork, what should I pay attention to?"

inputs3 = {
        "question": user_response_turn_3,
    }

print("\n===== Test Case 1: Second Turn - User Accepts Art Offer =====")
result3 = adaptive_rag.invoke(inputs3, config)
# --- Add this part ---
# Extract the relevant messages from the result state
rag_answer = result3.get("generation", "")
suggestion_offer = result3.get("suggestion_message", "")

# Combine and print the desired output for the user
full_response = f"{rag_answer}\n\n{suggestion_offer}".strip()
print("\n--- AI Response (Turn 3) ---")
print(full_response)
print("---------------------------\n")